# Gemma 4 vision-encoder pruning

This notebook presents the encoder-depth experiment with two simple figures:

1. **Quality versus vision latency** — the main trade-off plot.
2. **Distillation recovery** — how much quality a distilled 12/16 encoder recovers.

End-to-end latency is intentionally not the main axis because autoregressive LLM decoding dominates it. Results shown here use 10 BR test documents, so small quality differences should be confirmed on the complete test set.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
FIGURE_DIR = Path("results/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## Option A: load evaluation JSON files

This cell discovers the full-depth, untrained-pruning, and distilled `n10` evaluations under `results/evaluation/`.

In [ ]:
RESULT_COLUMNS = [
    "tokens",
    "layers",
    "total_layers",
    "micro_f1",
    "vision_ms",
    "end_to_end_ms",
    "model_type",
    "source",
]


def load_evaluation_results(directory=Path("results/evaluation")):
    rows = []
    pattern = "gemma4-e4b_*manual_BR_test_*imgtok_n10.json"
    for path in sorted(directory.glob(pattern)):
        data = json.loads(path.read_text(encoding="utf-8"))
        config = data["config"]
        if config.get("adapter"):
            continue
        pruning = config["vision_pruning"]
        layers = pruning["retained_layers"]
        total_layers = pruning["original_layers"]
        if config.get("vision_checkpoint"):
            model_type = "Distilled"
        elif layers == total_layers:
            model_type = "Full encoder"
        else:
            model_type = "Untrained pruning"
        rows.append(
            {
                "tokens": config["image_tokens"],
                "layers": layers,
                "total_layers": total_layers,
                "micro_f1": data["metrics"]["micro_f1"],
                "vision_ms": data["timing"]["mean_ms"]["vision_encoder_ms"],
                "end_to_end_ms": data["timing"]["mean_ms"]["end_to_end_ms"],
                "model_type": model_type,
                "source": path.name,
            }
        )
    return pd.DataFrame(rows, columns=RESULT_COLUMNS)


loaded_results = load_evaluation_results()
print(f"Loaded {len(loaded_results)} evaluations")
loaded_results.sort_values(["tokens", "layers", "model_type"])

## Option B: hardcoded results provided during the experiment

This cell makes the notebook usable without copying the evaluation JSON files. The 1120-token distilled end-to-end value was reported as approximately 7 seconds.

In [ ]:
hardcoded_results = pd.DataFrame(
    [
        # tokens, layers, total, F1, vision ms, end-to-end ms, type
        (280, 4, 16, 0.0250, 23.6, 7388.7, "Untrained pruning"),
        (280, 8, 16, 0.0250, 27.7, 6830.2, "Untrained pruning"),
        (280, 12, 16, 0.0250, 34.4, 7168.3, "Untrained pruning"),
        (280, 16, 16, 0.6667, 39.1, 7226.2, "Full encoder"),
        (560, 4, 16, 0.0250, 19.1, 7259.2, "Untrained pruning"),
        (560, 8, 16, 0.0250, 20.4, 7101.2, "Untrained pruning"),
        (560, 12, 16, 0.0500, 33.5, 6971.7, "Untrained pruning"),
        (560, 16, 16, 0.7000, 38.6, 7382.5, "Full encoder"),
        (1120, 4, 16, 0.0250, 25.0, 7779.6, "Untrained pruning"),
        (1120, 8, 16, 0.0250, 44.1, 6833.3, "Untrained pruning"),
        (1120, 12, 16, 0.1000, 52.8, 6849.0, "Untrained pruning"),
        (1120, 16, 16, 0.8000, 69.1, 7951.7, "Full encoder"),
        (560, 12, 16, 0.6750, 32.0, 6868.0, "Distilled"),
        (1120, 12, 16, 0.7000, 59.0, 7000.0, "Distilled"),
    ],
    columns=RESULT_COLUMNS[:-1],
)
hardcoded_results["source"] = "hardcoded"
hardcoded_results.sort_values(["tokens", "layers", "model_type"])

## Select the data source

Leave `USE_HARDCODED = True` to reproduce the figures from the reported values. Change it to `False` after placing the JSON files in `results/evaluation/`.

In [ ]:
USE_HARDCODED = True

if USE_HARDCODED:
    results = hardcoded_results.copy()
elif loaded_results.empty:
    raise ValueError("No evaluation JSON files were found")
else:
    results = loaded_results.copy()

results

## Main figure: quality–latency trade-off

Points toward the **upper-left** are better. Lines show direct block removal without distillation; stars show the distilled students.

In [ ]:
TOKEN_COLORS = {280: "#4C78A8", 560: "#F58518", 1120: "#54A24B"}
fig, ax = plt.subplots(figsize=(9, 6))

depth_sweep = results[results["model_type"].isin(["Full encoder", "Untrained pruning"])]
for tokens, group in depth_sweep.groupby("tokens"):
    group = group.sort_values("layers")
    color = TOKEN_COLORS.get(tokens)
    ax.plot(
        group["vision_ms"],
        group["micro_f1"],
        marker="o",
        linewidth=1.8,
        color=color,
        label=f"{tokens} tokens: direct pruning",
    )
    for row in group.itertuples():
        ax.annotate(
            f"{row.layers}/16",
            (row.vision_ms, row.micro_f1),
            xytext=(4, 5),
            textcoords="offset points",
            fontsize=8,
            color=color,
        )

distilled = results[results["model_type"] == "Distilled"]
for index, row in enumerate(distilled.itertuples()):
    ax.scatter(
        row.vision_ms,
        row.micro_f1,
        marker="*",
        s=240,
        color=TOKEN_COLORS.get(row.tokens),
        edgecolor="black",
        linewidth=0.8,
        zorder=5,
        label="Distilled student" if index == 0 else None,
    )
    ax.annotate(
        f"distilled 12/16\n{row.tokens} tokens",
        (row.vision_ms, row.micro_f1),
        xytext=(7, -20),
        textcoords="offset points",
        fontsize=9,
    )

ax.set(
    title="Gemma 4 encoder pruning: quality versus vision latency",
    xlabel="Mean vision-encoder latency (ms) — lower is better",
    ylabel="Micro F1 — higher is better",
)
ax.set_ylim(bottom=0)
ax.legend(frameon=True, loc="lower right")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "gemma4_pruning_tradeoff.png", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "gemma4_pruning_tradeoff.pdf", bbox_inches="tight")
plt.show()

## Supporting figure: distillation repairs pruning damage

The bar labels include Micro F1 and mean vision latency. This is the clearest figure for explaining the main finding.

In [ ]:
budgets = [560, 1120]
series = [
    ("Full 16/16", "Full encoder", 16, "#4C78A8"),
    ("Pruned 12/16\n(no distillation)", "Untrained pruning", 12, "#E45756"),
    ("Distilled 12/16", "Distilled", 12, "#54A24B"),
]
x = np.arange(len(budgets))
width = 0.24
fig, ax = plt.subplots(figsize=(8.5, 5.5))

for offset, (label, model_type, layers, color) in enumerate(series):
    values = []
    latencies = []
    for tokens in budgets:
        match = results[
            (results["tokens"] == tokens)
            & (results["model_type"] == model_type)
            & (results["layers"] == layers)
        ]
        if match.empty:
            values.append(np.nan)
            latencies.append(np.nan)
        else:
            values.append(match.iloc[0]["micro_f1"])
            latencies.append(match.iloc[0]["vision_ms"])
    positions = x + (offset - 1) * width
    bars = ax.bar(positions, values, width, label=label, color=color)
    for bar, f1, latency in zip(bars, values, latencies):
        if not np.isnan(f1):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.018,
                f"{f1:.3f}\n{latency:.1f} ms",
                ha="center",
                va="bottom",
                fontsize=9,
            )

ax.set_xticks(x, [f"{tokens} image tokens" for tokens in budgets])
ax.set_ylabel("Micro F1")
ax.set_ylim(0, 0.95)
ax.set_title("Representation distillation recovers a pruned Gemma 4 encoder")
ax.legend(frameon=True, loc="upper left")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "gemma4_distillation_recovery.png", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "gemma4_distillation_recovery.pdf", bbox_inches="tight")
plt.show()

## Presentation summary

The following cell generates concise numbers suitable for a slide caption.

In [ ]:
for tokens in budgets:
    full = results[
        (results.tokens == tokens) & (results.model_type == "Full encoder")
    ].iloc[0]
    distilled = results[
        (results.tokens == tokens) & (results.model_type == "Distilled")
    ].iloc[0]
    quality_retained = 100 * distilled.micro_f1 / full.micro_f1
    latency_reduction = 100 * (full.vision_ms - distilled.vision_ms) / full.vision_ms
    print(
        f"{tokens} tokens: the distilled 12/16 encoder retains "
        f"{quality_retained:.1f}% of full-depth F1 and reduces vision latency "
        f"by {latency_reduction:.1f}%."
    )

### Recommended message

- Removing encoder blocks without training causes catastrophic quality loss.
- Final-representation distillation recovers most of the lost extraction quality.
- At 560 tokens, the distilled 12/16 encoder retains about **96% of full-depth F1** while reducing vision latency by about **17%**.
- Vision encoding is below 1% of single-document end-to-end latency in this setup, so this optimization primarily improves encoder compute rather than total generation latency.
- These are preliminary results on 10 documents and should not be presented with error bars or statistical claims until evaluated on the full test set.